# 04. NLP 전처리 — 한국어 리뷰

## 목적
한국어 리뷰 텍스트를 정제하고 NLP 분석에 사용할 수 있는 형태로 변환한다.

## 영어(03) vs 한국어(04) 차이점
| 항목 | 영어 (03) | 한국어 (04) |
|------|----------|------------|
| 형태소 분석기 | spaCy (en_core_web_sm) | **Kiwi (kiwipiepy)** |
| 토큰화 방식 | 띄어쓰기 + 파이프라인 | 형태소 분석 (교착어 특성) |
| 원형 복원 | lemmatizer (별도 단계) | Kiwi 내장 (형태소 분리 시 자동) |
| 품사 태그 체계 | Universal POS (NOUN, VERB 등) | 세종 품사 태그 (NNG, VV 등) |
| 불용어 처리 | `token.is_stop` + 수동 필터 | **Kiwi Stopwords 클래스** + 품사 필터 |
| 배치 처리 | `nlp.pipe(texts, batch_size=500)` | `kiwi.tokenize(texts)` (리스트 → 자동 멀티스레드) |

## 분석 순서
1. 라이브러리 & DB 연결
2. 한국어 리뷰 로드 & 그룹 컬럼 생성
3. 기본 텍스트 정제 (URL, HTML, 특수문자 제거)
4. Kiwi 형태소 분석기 설정 (사용자 사전 + Stopwords)
5. 형태소 분석 + 토큰 추출 (멀티스레드 배치 처리)
6. 불용어 추가 제거 (게임 리뷰 특화)
7. 정제 결과 확인 (전체 / 시기별 / 유저 그룹별)
8. DB 저장
9. DB 연결 종료

---
## 1. 라이브러리 & DB 연결

In [ ]:
import sqlite3
import re
import pandas as pd
import numpy as np
from kiwipiepy import Kiwi
from kiwipiepy.utils import Stopwords
from collections import Counter
from tqdm import tqdm

tqdm.pandas()

DB_PATH = '../data/dave_diver.db'
conn    = sqlite3.connect(DB_PATH)

# ── Kiwi 형태소 분석기 로드 ──────────────────────────────────────────
# num_workers: 멀티스레드 워커 수 (0 = CPU 코어 수에 맞게 자동 설정)
# tokenize()에 리스트를 넣으면 자동으로 멀티스레드 배치 처리
kiwi = Kiwi(num_workers=4)

print(f'Kiwi 로드 완료')

Kiwi 로드 완료


Quantization is not supported for ArchType::neon. Fall back to non-quantized model.


### Kiwi 기본 개념

#### 핵심 메서드
| 메서드 | 설명 |
|--------|------|
| `kiwi.tokenize(text)` | 단일 str → 싱글스레드 처리, Token 리스트 반환 |
| `kiwi.tokenize(texts)` | **str의 Iterable → 멀티스레드 배치 처리**, Token 리스트의 iterator 반환 |
| `kiwi.add_user_word(word, tag, score)` | 사용자 사전에 단어 추가 |
| `Stopwords()` | 기본 불용어 사전 로드. `tokenize(stopwords=...)` 파라미터로 직접 전달 가능 |

#### tokenize() 주요 파라미터 
| 파라미터 | 기본값 | 설명 |
|---------|--------|------|
| `normalize_coda` | False | True면 '재밌엌ㅋㅋ' → '재밌어 ㅋㅋ'로 받침 정규화 |
| `split_complex` | False | True면 복합명사를 최대한 분할 ('고마움'→'고맙+음') |
| `stopwords` | None | Stopwords 객체를 주면 불용어 자동 제거 |
| `z_coda` | True | '먹었어욥' → '먹었어요 + ㅂ' 덧붙은 받침 분리 |

#### Token 객체 속성
| 속성 | 설명 | 예시 |
|------|------|------|
| `token.form` | 형태소 원형 | `"재밌"` |
| `token.tag` | 세종 품사 태그 | `"VA"` (형용사) |
| `token.start` | 원문에서의 시작 위치 | `0` |
| `token.len` | 형태소 길이 | `2` |

#### 세종 품사 태그 (이 프로젝트에서 사용)
| 태그 | 의미 | 영어 대응 | 예시 |
|------|------|----------|------|
| NNG | 일반명사 | NOUN | 게임, 스토리, 낚시 |
| NNP | 고유명사 | NOUN (proper) | 데이브, 반초 |
| VV | 동사 | VERB | 하다, 즐기다, 잡다 |
| VA | 형용사 | ADJ | 재밌다, 좋다, 예쁘다 |
| MAG | 일반부사 | ADV | 정말, 매우, 진짜 |

#### spaCy vs Kiwi 비교
```
영어 spaCy:  token.text / token.pos_  / token.lemma_ / token.is_stop
한국어 Kiwi: token.form / token.tag   / (form이 곧 원형) / Stopwords 클래스로 처리
```

한국어는 교착어이기 때문에 형태소 분석 자체가 원형 복원을 포함한다.  
예: "재밌었는데" → `재밌/VA` + `었/EP` + `는데/EC` → 형용사 "재밌" 추출  

**중요:** tag에 `-R`(규칙 활용), `-I`(불규칙 활용) 접미사가 붙을 수 있다.  
예: `VV-R`(규칙동사), `VA-I`(불규칙형용사) → 기본 태그만 추출해서 필터링해야 한다.

---
## 2. 한국어 리뷰 로드 & 그룹 컬럼 생성

03번 영어 노트북과 동일한 그룹 컬럼을 생성한다.
- `review_month` : 시기별 분석
- `play_segment` : 플레이타임 5구간
- `voted_up` : 긍정/부정
- `received_for_free` / `written_during_early_access`

In [ ]:
df_ko = pd.read_sql("""
    SELECT
        review_id,
        review_text,
        voted_up,
        playtime_at_review,
        received_for_free,
        written_during_early_access,
        review_month,
        language
    FROM reviews
    WHERE language = 'koreana'
      AND review_text IS NOT NULL
      AND TRIM(review_text) != ''
""", conn)

# 플레이타임 시간 단위 변환
df_ko['playtime_hours'] = df_ko['playtime_at_review'] / 60

# 플레이타임 5구간 생성 (03번과 동일 기준)
bins_p   = [0, 120, 600, 1800, 3000, 999999]
labels_p = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']
df_ko['play_segment'] = pd.cut(df_ko['playtime_at_review'], bins=bins_p, labels=labels_p)

print(f'한국어 리뷰 총 수          : {len(df_ko):,}건')
print(f'긍정                      : {df_ko["voted_up"].sum():,}건 ({df_ko["voted_up"].mean()*100:.1f}%)')
print(f'부정                      : {(df_ko["voted_up"]==0).sum():,}건')
print(f'무료 수령                 : {df_ko["received_for_free"].sum():,}건')
print(f'얼리액세스                : {df_ko["written_during_early_access"].sum():,}건')
print()
print('플레이타임 구간별 분포:')
print(df_ko['play_segment'].value_counts().sort_index())

한국어 리뷰 총 수          : 11,288건
긍정                      : 10,974건 (97.2%)
부정                      : 314건
무료 수령                 : 56건
얼리액세스                : 3,470건

플레이타임 구간별 분포:
play_segment
casual(<2h)         391
regular(2-10h)     3021
engaged(10-30h)    4468
engaged(30-50h)    1971
hardcore(50h+)     1437
Name: count, dtype: int64


---
## 3. 기본 텍스트 정제

Kiwi 처리 전에 노이즈를 제거한다.

제거 대상:
- URL (http, www)
- HTML 태그
- 특수문자 (한글, 영문, 숫자, 공백만 유지)
- 연속 공백

**영어(03)와 차이점:** 한국어는 숫자를 유지한다 ("10점 만점", "100시간" 등 맥락에 활용).
또한 영문 알파벳도 유지한다 (게임 용어: DLC, BGM, UI 등).

In [ ]:
def basic_clean_ko(text: str) -> str:
    """URL, HTML, 특수문자 제거 (한글+영문+숫자+공백 유지)"""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)       # URL 제거
    text = re.sub(r'<[^>]+>', '', text)                 # HTML 태그 제거
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', text)  # 한글+자모+영문+숫자+공백만 유지
    text = re.sub(r'\s+', ' ', text).strip()            # 연속 공백 제거
    return text

df_ko['text_cleaned'] = df_ko['review_text'].apply(basic_clean_ko)

# 정제 후 빈 텍스트 제거
before = len(df_ko)
df_ko  = df_ko[df_ko['text_cleaned'].str.strip() != ''].copy()
print(f'정제 후 빈 텍스트 제거: {before - len(df_ko):,}건')
print(f'남은 리뷰             : {len(df_ko):,}건')

# 정제 전후 비교 샘플
print('\n--- 정제 전후 비교 ---')
for _, row in df_ko.sample(5, random_state=42).iterrows():
    print(f'원문  : {row["review_text"][:120]}')
    print(f'정제후: {row["text_cleaned"][:120]}')
    print()

정제 후 빈 텍스트 제거: 130건
남은 리뷰             : 11,158건

--- 정제 전후 비교 ---
원문  : 진짜 잘만들었다.
연출이며 게임성이여 아이디어도 좋음
정제후: 진짜 잘만들었다 연출이며 게임성이여 아이디어도 좋음

원문  : 너무재밌어용
정제후: 너무재밌어용

원문  : 다 섞어 뒀는데 뭐 딱히 신경안쓸수 있게 해둬서 좋넹
정제후: 다 섞어 뒀는데 뭐 딱히 신경안쓸수 있게 해둬서 좋넹

원문  : 정성가득한 게임
정제후: 정성가득한 게임

원문  : ㄹㅇ 씹갓겜 정가로 사도 후회 절대 안함 할거 개많아서 오래할 수 있음
정제후: ㄹㅇ 씹갓겜 정가로 사도 후회 절대 안함 할거 개많아서 오래할 수 있음



---
## 4. Kiwi 형태소 분석기 설정

### 4-1. 사용자 사전 등록
게임 고유 용어를 사전에 추가한다. 등록하지 않으면 "데이브"가 "데이/NNG + 브/NNG"로 분리될 수 있다.

### 4-2. Kiwi 내장 Stopwords 설정
Kiwi는 `kiwipiepy.utils.Stopwords` 클래스를 내장하고 있어서,  
기본 불용어 사전을 로드하고 커스텀 불용어를 추가한 뒤  
`tokenize(stopwords=...)` 파라미터로 직접 전달할 수 있다.  
영어(03)에서 수동으로 `if not token.is_stop` 필터링한 것보다 깔끔하다.

In [ ]:
# ── 4-1. 사용자 사전 등록 ──────────────────────────────────────────
# add_user_word(word, tag='NNP', score=0.0)
# score가 높을수록 해당 단어로 분석될 확률이 높아짐
USER_WORDS = [
    ('데이브', 'NNP', 10.0),     # 고유명사: 게임 주인공
    ('반초', 'NNP', 10.0),       # 고유명사: NPC
    ('초밥집', 'NNG', 5.0),      # 일반명사: 게임 핵심 요소
    ('보스전', 'NNG', 5.0),      # 일반명사: 게임 요소
    ('엔딩', 'NNG', 5.0),        # 일반명사
    ('스토리', 'NNG', 5.0),      # 일반명사
    ('힐링', 'NNG', 5.0),        # 일반명사
    ('갓겜', 'NNG', 5.0),        # 신조어: 갓(God) + 게임
    ('꿀잼', 'NNG', 5.0),        # 신조어: 꿀 + 재미
    ('노잼', 'NNG', 5.0),        # 신조어: no + 재미
    ('갓작', 'NNG', 5.0),        # 신조어: 갓(God) + 작품
    ('명작', 'NNG', 5.0),        # 일반명사
    ('민트로켓', 'NNP', 10.0),   # 고유명사: 개발사
    ('넥슨', 'NNP', 10.0),       # 고유명사: 퍼블리셔
]

for word, tag, score in USER_WORDS:
    kiwi.add_user_word(word, tag, score)

print(f'사용자 사전 등록: {len(USER_WORDS)}개 단어')

# ── 4-2. Kiwi 내장 Stopwords 설정 ──────────────────────────────────
# Stopwords() : 기본 한국어 불용어 사전 로드
# .add() : 커스텀 불용어 추가 (str 또는 (형태소, 품사태그) 튜플)
stopwords = Stopwords()

print(f'\n기본 Stopwords 로드 완료')

사용자 사전 등록: 14개 단어

기본 Stopwords 로드 완료


In [ ]:
# 토큰화 테스트
test_text = '데이브 더 다이버 진짜 재밌었는데 보스전이 너무 어려웠다ㅋㅋㅋ'

# normalize_coda=True : '어려웠다ㅋㅋㅋ' 같은 받침 노이즈 처리
test_tokens = kiwi.tokenize(test_text, normalize_coda=True)

print(f'테스트 문장: "{test_text}"')
print(f'{"형태소":12s} {"품사":10s} 설명')
print('-' * 45)

TAG_DESC = {
    'NNG': '일반명사', 'NNP': '고유명사', 'NNB': '의존명사',
    'VV': '동사', 'VA': '형용사', 'VX': '보조용언', 'VCN': '부정지정사',
    'MAG': '일반부사', 'MAJ': '접속부사', 'MM': '관형사',
    'JKS': '주격조사', 'JKO': '목적격조사', 'JKB': '부사격조사', 'JX': '보조사',
    'EP': '선어말어미', 'EF': '종결어미', 'EC': '연결어미', 'ETM': '관형형전성어미',
    'XSV': '동사파생접미사', 'XSA': '형용사파생접미사',
    'SF': '마침표', 'SW': '특수문자', 'SN': '숫자',
}

for t in test_tokens:
    base_tag = t.tag.split('-')[0] if '-' in t.tag else t.tag
    desc = TAG_DESC.get(base_tag, '')
    print(f'{t.form:12s} {t.tag:10s} {desc}')

# stopwords 적용 테스트
print('\n--- stopwords 적용 후 ---')
test_tokens_filtered = kiwi.tokenize(test_text, normalize_coda=True, stopwords=stopwords)
for t in test_tokens_filtered:
    base_tag = t.tag.split('-')[0] if '-' in t.tag else t.tag
    desc = TAG_DESC.get(base_tag, '')
    print(f'{t.form:12s} {t.tag:10s} {desc}')

테스트 문장: "데이브 더 다이버 진짜 재밌었는데 보스전이 너무 어려웠다ㅋㅋㅋ"
형태소          품사         설명
---------------------------------------------
데이브          NNP        고유명사
더            NNP        고유명사
다이버          NNG        일반명사
진짜           MAG        일반부사
재밌           VA         형용사
었            EP         선어말어미
는데           EC         연결어미
보스전          NNG        일반명사
이            JKS        주격조사
너무           MAG        일반부사
어렵           VA-I       형용사
었            EP         선어말어미
다            EF         종결어미
ㅋㅋㅋ          SW         특수문자

--- stopwords 적용 후 ---
데이브          NNP        고유명사
더            NNP        고유명사
다이버          NNG        일반명사
진짜           MAG        일반부사
재밌           VA         형용사
는데           EC         연결어미
보스전          NNG        일반명사
너무           MAG        일반부사
어렵           VA-I       형용사


---
## 5. 형태소 분석 + 토큰 추출 (멀티스레드 배치 처리)

- 품사 필터: NNG(일반명사), NNP(고유명사), VV(동사), VA(형용사), MAG(일반부사)
- 1글자 토큰 제거 (한글 1글자는 대부분 의미가 약함)
- `normalize_coda=True` : 'ㅋㅋㅋ', '재밌엌' 등 구어체 받침 정규화
- `stopwords=stopwords` : Kiwi 내장 불용어 1차 제거

### 배치 처리 방식 (영어와의 차이)
- **영어 spaCy**: `nlp.pipe(texts, batch_size=500)` → batch_size 직접 설정
- **한국어 Kiwi**: `kiwi.tokenize(texts)` → 리스트를 넣으면 **자동 멀티스레드 배치 처리**
  - `num_workers=0`으로 초기화했으므로 CPU 코어 수에 맞게 자동 분배
  - 반환값은 iterator → tqdm으로 진행률 표시 가능

In [7]:
# 유지할 품사 태그
KEEP_TAGS = {'NNG', 'NNP', 'VV', 'VA', 'MAG'}

texts = df_ko['text_cleaned'].tolist()
tokenized = []

# kiwi.tokenize()에 리스트를 넣으면 자동 멀티스레드 배치 처리
# normalize_coda=True : '재밌엌ㅋㅋ' → '재밌어 ㅋㅋ' 정규화
# stopwords=stopwords : Kiwi 내장 불용어 자동 제거
for result in tqdm(
    kiwi.tokenize(texts, normalize_coda=True, stopwords=stopwords),
    total=len(texts),
    desc='Kiwi 처리 중'
):
    tokens = []
    for t in result:
        # 불규칙 활용 태그 정리: VV-R, VA-I 등에서 기본 태그만 추출
        base_tag = t.tag.split('-')[0] if '-' in t.tag else t.tag
        
        if (base_tag in KEEP_TAGS
            and len(t.form) >= 2):     # 1글자 제거
            tokens.append(t.form)
    
    tokenized.append(' '.join(tokens))

df_ko['cleaned_text'] = tokenized
print('\nKiwi 처리 완료')

Kiwi 처리 중: 100%|██████████| 11158/11158 [00:07<00:00, 1587.51it/s]


Kiwi 처리 완료


##### 예시
- 원문: "데이브 더 다이버 진짜 재밌었는데 보스전이 너무 어려웠다ㅋㅋㅋ"
- Kiwi 처리 후: `데이브/NNP`, `다이버/NNG`, `진짜/MAG`, `재밌/VA`, `보스전/NNG`, `어렵/VA`, `ㅋㅋㅋ/SW`
- 품사 필터 + 1글자 제거 → `"데이브 다이버 진짜 재밌 보스전 어렵"`

조사(이, 을, 는), 어미(었, 는데, 다)는 품사 필터에서 자동 제거된다.

---
## 6. 불용어 추가 제거

Kiwi Stopwords + 품사 필터로 조사/어미/기본 불용어는 이미 제거되었지만,
게임 리뷰 특화 불용어를 추가로 제거한다.

- 게임 제목 자체 (데이브, 다이버)
- 리뷰 형식어 (게임, 추천, 플레이, 리뷰)
- 단독 분석 가치 낮은 용언 원형 (하다→하, 되다→되, 있다→있 등)

**참고:** Kiwi는 형태소 분석 시 '좋았다'→'좋/VA', '재밌었다'→'재밌/VA'로  
원형을 추출하므로, 불용어도 원형으로 등록해야 한다.

In [8]:
CUSTOM_STOPWORDS_KO = {
    # 게임 제목
    '데이브', '다이버', '다이브',
    # 리뷰 형식어
    '게임', '추천', '플레이', '리뷰', '시간', '정도', '진행',
    # 범용 용언 원형 (Kiwi가 추출하는 형태)
    '하다', '되다', '있다', '없다', '같다', '보다', '주다', '오다', '가다',
    # 원형이 짧게 추출되는 경우도 대비
    '하', '되', '있', '없', '같', '보', '주', '오', '가',
    '좋다', '나쁘다', '많다', '적다',
    '좋', '나쁘', '많', '적',
    # 범용 부사/명사
    '정말', '진짜', '매우', '너무', '아주', '완전', '그냥', '좀', '약간',
    '것', '수', '때', '점', '거', '중', '뭐', '더',
}

def remove_custom_stopwords_ko(text: str) -> str:
    tokens = [t for t in text.split() if t not in CUSTOM_STOPWORDS_KO]
    return ' '.join(tokens)

def deduplicate_tokens(text: str) -> str:
    """리뷰 내 동일 토큰 반복을 제거 (순서 유지, 각 토큰 최대 1번)"""
    seen = set()
    result = []
    for t in text.split():
        if t not in seen:
            seen.add(t)
            result.append(t)
    return ' '.join(result)

df_ko['cleaned_text'] = df_ko['cleaned_text'].apply(deduplicate_tokens)

df_ko['cleaned_text'] = df_ko['cleaned_text'].apply(remove_custom_stopwords_ko)

# 불용어 제거 후 빈 텍스트 처리
before = len(df_ko)
df_ko  = df_ko[df_ko['cleaned_text'].str.strip() != ''].copy()
print(f'불용어 제거 후 빈 텍스트 제거: {before - len(df_ko):,}건')
print(f'최종 한국어 리뷰 수         : {len(df_ko):,}건')

# 최종 샘플
print('\n--- 최종 정제 결과 샘플 ---')
for _, row in df_ko.sample(4, random_state=42).iterrows():
    label = '긍정' if row['voted_up'] else '부정'
    print(f'[{label} / {row["play_segment"]}]')
    print(f'  원문  : {row["review_text"][:100]}')
    print(f'  정제후: {row["cleaned_text"][:100]}')
    print()

불용어 제거 후 빈 텍스트 제거: 1,084건
최종 한국어 리뷰 수         : 10,074건

--- 최종 정제 결과 샘플 ---
[긍정 / engaged(10-30h)]
  원문  : 재밌다
  정제후: 재밌

[긍정 / engaged(10-30h)]
  원문  : 이 게임하고 바로 초밥 배달시킴
  정제후: 바로 초밥 배달

[긍정 / regular(2-10h)]
  원문  : 빨리 엔딩 문열어!!! 현기증 날꺼같아!!빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!
  정제후: 빨리 엔딩 현기증 나오 주말

[긍정 / engaged(10-30h)]
  원문  : 재밌음 야미
  정제후: 재밌 야미



```
[긍정 / regular(2-10h)]
  원문  : 빨리 엔딩 문열어!!! 현기증 날꺼같아!!빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!! 현기증 날꺼같아 빨리 엔딩 문열어!!
  정제후: 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 빨리 엔딩 현기증 
```

과같이 중복된 도배형식 리뷰가 있으면 빈도수가 왜곡될수 있기때문에 동일한 토큰을 제거했다.

---
## 7. 정제 결과 확인

### 7-1. 전체 상위 토큰
### 7-2. 긍정 vs 부정 상위 토큰 비교
### 7-3. 시기별 상위 토큰 비교 (groupby review_month)
### 7-4. 플레이타임 구간별 상위 토큰 비교 (groupby play_segment)

In [ ]:
# 7-1. 전체 상위 토큰
all_tokens = ' '.join(df_ko['cleaned_text']).split()
token_freq = Counter(all_tokens)

print(f'고유 토큰 수: {len(token_freq):,}개')
print(f'전체 토큰 수: {len(all_tokens):,}개')
print()
print('전체 상위 30 토큰:')
for word, cnt in token_freq.most_common(30):
    print(f'  {word:20s} {cnt:,}')

고유 토큰 수: 8,035개
전체 토큰 수: 91,630개

전체 상위 30 토큰:
  재밌                   2,678
  만들                   1,002
  넥슨                   973
  스토리                  844
  컨텐츠                  836
  재미있                  815
  나오                   768
  재미                   684
  힐링                   669
  모르                   648
  갓겜                   642
  물고기                  631
  생각                   615
  즐기                   602
  정식                   588
  도트                   468
  출시                   461
  바다                   437
  초밥                   410
  엔딩                   406
  기대                   401
  느낌                   399
  느끼                   393
  요소                   384
  많이                   382
  계속                   382
  꿀잼                   377
  운영                   372
  다양                   355
  그래픽                  353


In [ ]:
# 7-2. 긍정 vs 부정 상위 토큰 비교
for label, val in [('긍정', 1), ('부정', 0)]:
    tokens = ' '.join(df_ko[df_ko['voted_up'] == val]['cleaned_text']).split()
    print(f'--- {label} 리뷰 상위 20 토큰 ---')
    for w, c in Counter(tokens).most_common(20):
        print(f'  {w:20s} {c:,}')
    print()

--- 긍정 리뷰 상위 20 토큰 ---
  재밌                   2,634
  만들                   955
  넥슨                   951
  재미있                  802
  컨텐츠                  801
  스토리                  796
  나오                   727
  힐링                   660
  갓겜                   638
  재미                   638
  모르                   608
  물고기                  597
  즐기                   587
  생각                   583
  정식                   578
  도트                   460
  출시                   455
  바다                   422
  초밥                   397
  기대                   392

--- 부정 리뷰 상위 20 토큰 ---
  스토리                  48
  만들                   47
  재미                   46
  재밌                   44
  나오                   41
  모르                   40
  컨텐츠                  35
  물고기                  34
  생각                   32
  마을                   27
  느낌                   26
  퀘스트                  26
  미니                   25
  처음                   23
  계속                   23
  넥슨                 

In [ ]:
# 7-3. 시기별 상위 토큰 비교
target_months = ['2023-06', '2023-08', '2023-12', '2024-05']

print('=== 시기별 상위 15 토큰 ===')
for month in target_months:
    sub = df_ko[df_ko['review_month'] == month]
    if len(sub) == 0:
        continue
    tokens = ' '.join(sub['cleaned_text']).split()
    top    = Counter(tokens).most_common(15)
    print(f'\n[{month}] n={len(sub):,}건')
    print(', '.join([f'{w}({c})' for w, c in top]))

=== 시기별 상위 15 토큰 ===

[2023-06] n=370건
재밌(92), 만들(36), 넥슨(33), 출시(31), 나오(30), 정식(30), 재미있(27), 모르(22), 갓겜(21), 스토리(21), 힐링(21), 컨텐츠(20), 재미(19), 빨리(16), 초밥(16)

[2023-08] n=350건
재밌(106), 스토리(44), 나오(42), 컨텐츠(42), 재미있(35), 생각(33), 만들(32), 힐링(31), 넥슨(29), 즐기(28), 엔딩(28), 재미(26), 물고기(24), 갓겜(23), 요소(22)

[2023-12] n=341건
재밌(91), 만들(37), 즐기(31), 재미(31), 재미있(29), 힐링(27), 컨텐츠(23), 나오(22), 넥슨(21), 스토리(21), 생각(20), 모르(20), 갓겜(17), 타이(16), 물고기(16)

[2024-05] n=132건
재밌(39), 스토리(15), 재미(12), 컨텐츠(12), 넥슨(10), 엔딩(10), 만들(10), 물고기(10), 힐링(9), 무료(9), 그래픽(9), 나오(9), 재미있(8), 느낌(8), 요소(8)


In [ ]:
# 7-4. 플레이타임 구간별 부정 리뷰 상위 토큰 비교
print('=== 플레이타임 구간별 부정 리뷰 상위 15 토큰 ===')
for seg in labels_p:
    sub = df_ko[(df_ko['play_segment'] == seg) & (df_ko['voted_up'] == 0)]
    if len(sub) < 10:
        continue
    tokens = ' '.join(sub['cleaned_text']).split()
    top    = Counter(tokens).most_common(15)
    print(f'\n[{seg}] 부정 n={len(sub):,}건')
    print(', '.join([f'{w}({c})' for w, c in top]))

=== 플레이타임 구간별 부정 리뷰 상위 15 토큰 ===

[casual(<2h)] 부정 n=38건
환불(8), 모르(7), 재미(7), 넥슨(4), 주인공(4), 긍정(4), 재미없(4), 화면(4), 미니(4), 시작(3), 만들(3), 평가(3), 압도(3), 역시(3), 수준(3)

[regular(2-10h)] 부정 n=63건
재미(10), 재밌(10), 물고기(8), 만들(8), 스토리(7), 계속(7), 퀘스트(7), 초반(7), 넥슨(6), 반복(6), 나오(5), 다시(5), 바다(5), 마을(5), 느낌(4)

[engaged(10-30h)] 부정 n=133건
스토리(28), 만들(25), 모르(24), 컨텐츠(22), 재밌(22), 나오(22), 재미(21), 생각(20), 처음(18), 물고기(17), 미니(16), 마을(16), 퀘스트(16), 보스전(15), 느낌(15)

[engaged(30-50h)] 부정 n=38건
스토리(8), 재밌(7), 즐기(5), 생각(4), 경영(4), 물고기(4), 자꾸(4), 가게(3), 운영(3), 느낌(3), 요소(3), 만들(3), 기분(3), 모르(3), 나오(3)

[hardcore(50h+)] 부정 n=26건
나오(9), 만들(8), 컨텐츠(6), 생각(6), 재미(5), 개선(5), 계속(4), 많이(4), 초반(4), 느리(4), 들어가(4), 모르(4), 구매(4), 판매(4), 시스템(4)


---
## 8. DB 저장

정제된 텍스트와 그룹 컬럼을 `cleaned_reviews_ko` 테이블로 저장.
원본 reviews 테이블은 수정하지 않는다.

In [ ]:
save_cols = [
    'review_id', 'voted_up', 'review_month', 'play_segment',
    'playtime_at_review', 'playtime_hours',
    'received_for_free', 'written_during_early_access',
    'review_text', 'cleaned_text'
]

df_ko[save_cols].to_sql(
    'cleaned_reviews_ko',
    conn,
    if_exists='replace',
    index=False
)

# 저장 확인
result = pd.read_sql("SELECT COUNT(*) as total FROM cleaned_reviews_ko", conn)
schema = pd.read_sql("PRAGMA table_info(cleaned_reviews_ko)", conn)

print(f'저장 완료: {result["total"][0]:,}건')
print()
print('저장된 컬럼:')
print(schema[['name', 'type']].to_string(index=False))

# groupby 활용 예시
print('\n--- groupby 활용 예시 ---')
print('시기별 리뷰 수 확인:')
check = pd.read_sql("""
    SELECT review_month, COUNT(*) as cnt,
           ROUND(AVG(voted_up)*100,1) as pos_rate
    FROM cleaned_reviews_ko
    GROUP BY review_month
    ORDER BY review_month
    LIMIT 10
""", conn)
print(check.to_string(index=False))

저장 완료: 10,074건

저장된 컬럼:
                       name    type
                  review_id    TEXT
                   voted_up INTEGER
               review_month    TEXT
               play_segment    TEXT
         playtime_at_review INTEGER
             playtime_hours    REAL
          received_for_free INTEGER
written_during_early_access INTEGER
                review_text    TEXT
               cleaned_text    TEXT

--- groupby 활용 예시 ---
시기별 리뷰 수 확인:
review_month  cnt  pos_rate
     2022-10  228      95.6
     2022-11 1788      98.8
     2022-12  333      97.9
     2023-01  171      95.9
     2023-02  109      98.2
     2023-03  234      97.9
     2023-04  131      94.7
     2023-05   86      96.5
     2023-06  370      97.8
     2023-07 1905      97.0


In [ ]:
"""
플레이타임 구간별 한국어 부정 리뷰 원문 확인
─────────────────────────────────────────
블로그에 쓴 해석이 실제 리뷰와 맞는지 검증용
"""
# ── 한국어 부정 리뷰 + 플레이타임 구간 로드 ──
df = pd.read_sql("""
    SELECT 
        review_id,
        review_text,
        playtime_at_review,
        CASE
            WHEN playtime_at_review < 120  THEN 'casual(<2h)'
            WHEN playtime_at_review < 600  THEN 'regular(2-10h)'
            WHEN playtime_at_review < 1800 THEN 'engaged(10-30h)'
            WHEN playtime_at_review < 3000 THEN 'engaged(30-50h)'
            ELSE 'hardcore(50h+)'
        END as segment,
        ROUND(playtime_at_review / 60.0, 1) as hours
    FROM reviews
    WHERE language = 'koreana'
      AND voted_up = 0
    ORDER BY playtime_at_review
""", conn)

conn.close()

# ── 구간별 출력 ──
segments = [
    ('casual(<2h)',      ['환불', '모르', '재미없', '화면', '긍정', '압도']),
    ('regular(2-10h)',   ['반복', '계속', '초반', '퀘스트']),
    ('engaged(10-30h)',  ['스토리', '보스전', '노가다', '마을', '미니', '퀘스트']),
    ('engaged(30-50h)', ['경영', '가게', '운영', '엔딩']),
    ('hardcore(50h+)',   ['개선', '시스템', '판매', '구매']),
]

for seg_name, keywords in segments:
    seg_df = df[df['segment'] == seg_name]
    
    print(f'\n{"="*80}')
    print(f'  {seg_name}  |  부정 리뷰 {len(seg_df)}건')
    print(f'{"="*80}')
    
    # 키워드 포함 리뷰 필터
    for kw in keywords:
        matched = seg_df[seg_df['review_text'].str.contains(kw, na=False)]
        print(f'\n--- "{kw}" 포함 리뷰 ({len(matched)}건) ---')
        for _, row in matched.head(5).iterrows():
            text = row['review_text'].replace('\n', ' ')[:200]
            print(f'  [{row["hours"]}h] {text}')
    
    # 키워드 무관하게 전체 리뷰도 몇 건 출력
    print(f'\n--- 전체 샘플 (최대 5건) ---')
    for _, row in seg_df.head(5).iterrows():
        text = row['review_text'].replace('\n', ' ')[:200]
        print(f'  [{row["hours"]}h] {text}')


  casual(<2h)  |  부정 리뷰 41건

--- "환불" 포함 리뷰 (8건) ---
  [0.2h] 환불 신청한지가 언젠디 아직도 환불이 안되고 있는거죠???
  [0.4h] 이 게임은 무척 매력적입니다. 하지만 아무리 보기 좋은 떡이어도 먹을 수 없다면 말짱 도루묵이죠. 수려한 도트와 듣기 좋은 브금까지 갖춘 이 게임의 단점은 바로, 유저가 키보드 셋팅을 할 자유가 아예 없단 것입니다.  이 게임에서는 마우스와 키보드를 함께 사용하여 플레이를 해야만 하죠. 그러나 저는 게임 플레이 시 마우스 조작이 익숙치 않은 편입니다. (아
  [0.5h] 이미 짜여져 있는 일직선상 스토리라인에 어거지로 따라가야 하는 느낌. 이런 느낌 싫어하는 사람이면 비추천임. 잠 와서 30분만에 환불,
  [1.1h] 작살난다냥 + 초밥 타이쿤. 근데 이제 캐릭터가 흉측한.. 사실 못생긴 거 못 참아서 환불한 것도 있음. 대체 왜 수염돼지가 주인공인데요
  [1.2h] 알바새기들인가 드럽게 재미없는 이걸 잼있다고 참나 한시간하고 환불

--- "모르" 포함 리뷰 (5건) ---
  [0.2h] 압도적 긍정적이라길래 해봤는데 20년전에 2G 폰으로 하던 게임 수준이다 그당시엔 할게 그런것밖에 없었고 게임 경험도 적었으니 그정도라도 만족했지만 할 게임이 많아지고, 온갖 컨텐츠가 포함된 게임을 많이 겪다보니 이건 뭐..  초딩들이나 좋아할법한 단순한 게임이라고 본다  것보다 물고기를 잡으려고 작살을 몇번 쏘는데 벌써부터 너무나 질리더라 너무나도 식상하
  [1.1h] 왜 평가가 압도적으로 긍정적인지 모르겠는 게임 1시간도 안돼서 질림  퀘스트나 할 게 너무 많고 정리가 안됨 순서가 뒤죽박죽 진짜모르겠다
  [1.6h] 일단.. 게임은 열심히 만들고, 어느정도 할만한 것 같지만, 게임 내용 외 측면에서 평가 자체는 비추.  애초에 맥에서는 실행도 안되는데 왜 지원하고 있는지 모르겠음.. 관련해서 요청하는대로 공식루트로 버그리포트 및 로그를 보내봐도 반응이 없어서,  문제를 인지하

---
## 9. DB 연결 종료

In [ ]:
conn.close()
print('DB 연결 종료')

DB 연결 종료
